In [1]:
import pandas as pd, folium
stops = pd.read_csv("../data/raw/delhi_bus_gtfs/stops.txt")
routes = pd.read_csv("../data/raw/delhi_bus_gtfs/routes.txt")
print(len(stops), "stops,", len(routes), "routes")
stops.head()

6342 stops, 2964 routes


,stop_id,stop_code,stop_name,stop_lat,stop_lon,zone_id
0,0,DC4539,Narela Terminal,28.852004,77.088059,0.0
1,1,DC4540,Police Station Narela,28.853366,77.088535,1.0
2,2,DC4541,Safiyabad Crossing,28.855266,77.089730,2.0
3,3,DC4542,Ramdev Chowk Pithori Jhori,28.857752,77.092856,3.0
4,4,DC4543,Narela A-6 / CPJ College,28.857652,77.095746,4.0


In [3]:
m = folium.Map(location=[28.61, 77.21], zoom_start=11, tiles="OpenStreetMap")
for _, s in stops.sample(500, random_state=0).iterrows():
    folium.CircleMarker([s.stop_lat, s.stop_lon], radius=2).add_to(m)
m

In [4]:
import glob
f = sorted(glob.glob("../data/raw/bus_positions_*.csv"))[-1]
pos = pd.read_csv(f).drop_duplicates(["vehicle_id", "timestamp"])
print(len(pos), "rows,", pos.vehicle_id.nunique(), "buses")

# pick a bus that has actually moved
g = pos.groupby("vehicle_id")
spread = (g.lat.max() - g.lat.min()) + (g.lon.max() - g.lon.min())
bus = spread[g.size() >= 10].sort_values(ascending=False).index[5]
trace = pos[pos.vehicle_id == bus].sort_values("timestamp")
print(bus, len(trace), "points")

m2 = folium.Map(location=[trace.lat.mean(), trace.lon.mean()], zoom_start=14, tiles="OpenStreetMap")
folium.PolyLine(trace[["lat", "lon"]].values.tolist(), weight=4).add_to(m2)

pad = 0.005
near = stops[stops.stop_lat.between(trace.lat.min() - pad, trace.lat.max() + pad)
             & stops.stop_lon.between(trace.lon.min() - pad, trace.lon.max() + pad)]
for _, s in near.iterrows():
    folium.CircleMarker([s.stop_lat, s.stop_lon], radius=3, color="red").add_to(m2)
m2

97464 rows, 5930 buses
DL1PD8224 11 points


In [ ]:
import numpy as np

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2)**2
    return 2 * 6371 * np.arcsin(np.sqrt(a))

def remove_jumps(t, max_km=1.0):
    t = t.sort_values("timestamp")
    med_lat = t.lat.rolling(5, center=True, min_periods=1).median()
    med_lon = t.lon.rolling(5, center=True, min_periods=1).median()
    dist = haversine_km(t.lat, t.lon, med_lat, med_lon)
    return t[dist < max_km]

clean = remove_jumps(trace)
print(f"kept {len(clean)} of {len(trace)} points")

m3 = folium.Map(location=[clean.lat.mean(), clean.lon.mean()], zoom_start=14, tiles="OpenStreetMap")
folium.PolyLine(clean[["lat", "lon"]].values.tolist(), weight=4).add_to(m3)
for _, s in near.iterrows():
    folium.CircleMarker([s.stop_lat, s.stop_lon], radius=3, color="red").add_to(m3)
m3